# ADNI Large SDF Visualization

This notebook inspects generated `.npz` SDF samples for a chosen preprocessing output root and group.

It provides:
- dataset-level file/count checks
- per-case label and mesh diagnostics
- 3D visualization of positive and negative SDF samples with mesh overlay
- SDF histograms
- an optional cell to compare two reruns for exact reproducibility

Default paths point to the 10-shape pilot under `tmp/adni_large_pilot`. Update `OUTPUT_ROOT` when you switch to the full dataset.

Reproducibility note: `src/PreprocessMesh.cpp` seeds sampling from `std::random_device`, so rerunning preprocessing does **not** produce bitwise-identical `.npz` files unless the C++ sampler is changed to use a fixed seed.


In [1]:
from pathlib import Path
import csv
import hashlib
import random

import numpy as np
import torch
import trimesh
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
OUTPUT_ROOT = Path("/home/jakaria/INR/Deep3DComp/tmp/adni_large_pilot")
GROUP = "left"  # left, right, combined
POINT_LIMIT = 12000
SHOW_MESH = True
RANDOM_SEED = 7
STEM = None  # Example: "1077_bl_left"


In [3]:
def group_paths(output_root: Path, group: str) -> dict:
    base = output_root / f"{group}_hippocampus_correspondence"
    return {
        "base": base,
        "npz_dir": base / "sdf_data" / "SdfSamples" / "minimal_scaled_obj_files",
        "obj_dir": base / "minimal_scaled_obj_files",
        "final_obj_dir": base / "minimal_final_obj",
        "final_ply_dir": base / "minimal_final_ply",
        "labels_path": base / "minimal_scaled_obj_files" / "labels.pt",
        "manifest_path": output_root / "manifests" / "selected_scans.csv",
        "validate_summary_path": output_root / "reports" / "validate_summary.json",
        "sdf_summary_path": output_root / "reports" / "sdf_summary.json",
    }


def read_manifest_rows(path: Path) -> list[dict]:
    with path.open() as f:
        return list(csv.DictReader(f))


def manifest_index(rows: list[dict], group: str) -> dict[str, dict]:
    key = f"{group}_stem"
    return {row[key]: row for row in rows}


def load_labels(path: Path) -> dict:
    return torch.load(path, map_location="cpu")


def available_stems(npz_dir: Path) -> list[str]:
    return sorted(path.stem for path in npz_dir.glob("*.npz"))


def subsample(points: np.ndarray, limit: int, seed: int) -> np.ndarray:
    if len(points) <= limit:
        return points
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(points), size=limit, replace=False)
    return points[idx]


def load_case(paths: dict, stem: str) -> dict:
    npz = np.load(paths["npz_dir"] / f"{stem}.npz")
    mesh = trimesh.load(paths["obj_dir"] / f"{stem}.obj", force="mesh", process=False)
    pos = npz["pos"]
    neg = npz["neg"]
    return {
        "stem": stem,
        "pos": pos,
        "neg": neg,
        "mesh": mesh,
        "bounds_min": mesh.bounds[0],
        "bounds_max": mesh.bounds[1],
        "mesh_volume": abs(float(mesh.volume)),
        "surface_area": float(mesh.area),
        "watertight": bool(mesh.is_watertight),
        "vertices": int(len(mesh.vertices)),
        "faces": int(len(mesh.faces)),
    }


def summarize_case(paths: dict, manifest_by_stem: dict, labels: dict, stem: str) -> dict:
    case = load_case(paths, stem)
    row = manifest_by_stem.get(stem, {})
    label = labels.get(stem)
    summary = {
        "stem": stem,
        "pos_count": int(case["pos"].shape[0]),
        "neg_count": int(case["neg"].shape[0]),
        "mesh_volume_scaled_units": case["mesh_volume"],
        "surface_area_scaled_units": case["surface_area"],
        "watertight": case["watertight"],
        "vertices": case["vertices"],
        "faces": case["faces"],
        "mesh_bounds_min": case["bounds_min"].tolist(),
        "mesh_bounds_max": case["bounds_max"].tolist(),
        "pos_sdf_range": [float(case["pos"][:, 3].min()), float(case["pos"][:, 3].max())],
        "neg_sdf_range": [float(case["neg"][:, 3].min()), float(case["neg"][:, 3].max())],
    }
    if row:
        summary.update(
            {
                "rid": row.get("RID"),
                "scan_id": row.get("scan_id"),
                "age_numeric": float(row.get("age_numeric", "nan")),
                "gender_numeric": float(row.get("gender_numeric", "nan")),
                "diagnosis_numeric": float(row.get("diagnosis_numeric", "nan")),
                "mask_volume_mm3": float(row.get(f"{GROUP}_mask_volume_mm3", "nan")),
                "mesh_volume_mm3": float(row.get(f"{GROUP}_mesh_volume_mm3", "nan")),
                "source_ply": row.get(f"{GROUP}_source_ply"),
            }
        )
    if label is not None:
        summary["label_vector"] = [float(x) for x in label.tolist()]
    return summary


def make_sdf_figure(paths: dict, stem: str, point_limit: int = 12000, show_mesh: bool = True, seed: int = 7):
    case = load_case(paths, stem)
    pos = subsample(case["pos"], point_limit, seed)
    neg = subsample(case["neg"], point_limit, seed + 1)
    mesh = case["mesh"]

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "xy"}]],
        column_widths=[0.72, 0.28],
        subplot_titles=(f"3D samples and mesh: {stem}", "SDF histogram"),
    )

    if show_mesh:
        fig.add_trace(
            go.Mesh3d(
                x=mesh.vertices[:, 0],
                y=mesh.vertices[:, 1],
                z=mesh.vertices[:, 2],
                i=mesh.faces[:, 0],
                j=mesh.faces[:, 1],
                k=mesh.faces[:, 2],
                color="lightgray",
                opacity=0.18,
                name="mesh",
                showscale=False,
            ),
            row=1,
            col=1,
        )

    fig.add_trace(
        go.Scatter3d(
            x=pos[:, 0],
            y=pos[:, 1],
            z=pos[:, 2],
            mode="markers",
            marker=dict(size=2, color=pos[:, 3], colorscale="Blues", opacity=0.55),
            name=f"pos ({len(pos)})",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            x=neg[:, 0],
            y=neg[:, 1],
            z=neg[:, 2],
            mode="markers",
            marker=dict(size=2, color=neg[:, 3], colorscale="Reds", opacity=0.55),
            name=f"neg ({len(neg)})",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Histogram(x=case["pos"][:, 3], nbinsx=80, name="pos sdf", marker_color="#4c78a8", opacity=0.7),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Histogram(x=case["neg"][:, 3], nbinsx=80, name="neg sdf", marker_color="#e45756", opacity=0.7),
        row=1,
        col=2,
    )

    fig.update_layout(
        barmode="overlay",
        height=720,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=45, b=0),
    )
    fig.update_scenes(aspectmode="data")
    fig.update_xaxes(title_text="SDF value", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    return fig


def compare_npz(npz_a: Path, npz_b: Path) -> dict:
    a = np.load(npz_a)
    b = np.load(npz_b)
    result = {}
    for key in ["pos", "neg"]:
        arr_a = a[key]
        arr_b = b[key]
        result[key] = {
            "shape_a": tuple(arr_a.shape),
            "shape_b": tuple(arr_b.shape),
            "exact_equal": bool(arr_a.shape == arr_b.shape and np.array_equal(arr_a, arr_b)),
            "sha256_a": hashlib.sha256(arr_a.tobytes()).hexdigest(),
            "sha256_b": hashlib.sha256(arr_b.tobytes()).hexdigest(),
        }
    return result


In [4]:
paths = group_paths(OUTPUT_ROOT, GROUP)
rows = read_manifest_rows(paths["manifest_path"])
rows_by_stem = manifest_index(rows, GROUP)
labels = load_labels(paths["labels_path"])
stems = available_stems(paths["npz_dir"])

print(f"output_root: {OUTPUT_ROOT}")
print(f"group: {GROUP}")
print(f"npz_dir: {paths['npz_dir']}")
print(f"npz_count: {len(stems)}")
print(f"label_count: {len(labels)}")
print(f"manifest_rows: {len(rows)}")
print("first stems:", stems[:10])

if STEM is None:
    STEM = stems[0]
    print(f"selected default STEM: {STEM}")


output_root: /home/jakaria/INR/Deep3DComp/tmp/adni_large_pilot
group: left
npz_dir: /home/jakaria/INR/Deep3DComp/tmp/adni_large_pilot/left_hippocampus_correspondence/sdf_data/SdfSamples/minimal_scaled_obj_files
npz_count: 10
label_count: 10
manifest_rows: 10
first stems: ['1077_bl_left', '1307_bl_left', '1346_bl_left', '158_bl_left', '2392_bl_left', '301_bl_left', '4004_bl_left', '5126_bl_left', '66_bl_left', '679_bl_left']
selected default STEM: 1077_bl_left


In [5]:
summary = summarize_case(paths, rows_by_stem, labels, STEM)
summary


{'stem': '1077_bl_left',
 'pos_count': 290383,
 'neg_count': 209032,
 'mesh_volume_scaled_units': 0.2572685510914217,
 'surface_area_scaled_units': 3.090217834359501,
 'watertight': True,
 'vertices': 2298,
 'faces': 4592,
 'mesh_bounds_min': [-0.46179098, -0.46338264, -0.88392444],
 'mesh_bounds_max': [0.46204474, 0.38948166, 0.9],
 'pos_sdf_range': [7.081689545884728e-08, 1.3373985290527344],
 'neg_sdf_range': [-0.18455485999584198, -2.2020867618266493e-07],
 'rid': '1077',
 'scan_id': '1077_bl',
 'age_numeric': 83.3,
 'gender_numeric': 0.0,
 'diagnosis_numeric': -1.0,
 'mask_volume_mm3': 3614.0,
 'mesh_volume_mm3': 3271.044621563943,
 'source_ply': '/home/jakaria/ADNI/ADNI_1_GO_Large/adni_hipp_ply_surfs/1077_bl.L.hipp.ply',
 'label_vector': [-1.0, 83.30000305175781, 0.0, 3614.0, 3271.044677734375]}

In [6]:
fig = make_sdf_figure(paths, STEM, point_limit=POINT_LIMIT, show_mesh=SHOW_MESH, seed=RANDOM_SEED)
fig.show()


In [7]:
# Optional reproducibility comparison.
# Set NPZ_A and NPZ_B to two reruns of the same mesh if you want to confirm
# that preprocessing is not bitwise deterministic.

NPZ_A = None
NPZ_B = None

if NPZ_A and NPZ_B:
    compare_npz(Path(NPZ_A), Path(NPZ_B))
else:
    print("Set NPZ_A and NPZ_B to compare two reruns.")


Set NPZ_A and NPZ_B to compare two reruns.
